In [8]:
!pip install scikit-image

  Using cached imageio-2.37.0-py3-none-any.whl.metadata (5.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 7.9 MB/s eta 0:00:0000:01:00:01
Using cached imageio-2.37.0-py3-none-any.whl (315 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-image] [scikit-image]


In [2]:
# JupyterLab Check Harness — с агрегатором всех запусков и копированием работ
# ----------------------------------------------------------------------------
from pathlib import Path
import os, sys, json, re, csv, time, subprocess, traceback, shutil
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple

# ==== ⚙️ НАСТРОЙКИ ====
TARGET_DIR = Path("data")               # Папка с работами
GLOB_PATTERNS = "*.py,*.ipynb"           # Маски файлов
DEFAULT_TIMEOUT = 300                    # Таймаут на файл (сек)
STOP_ON_FAIL = False                     # Остановиться на первом падении
REPORTS_ROOT = Path("reports")           # Корень отчётов
SAVE_LOGS = True                         # Писать stdout/stderr
VERBOSITY = 1                            # 0..2

# Имена агрегаторов в корне REPORTS_ROOT
ALL_SUMMARY_CSV = "summary_all_runs.csv"
ALL_GRADES_CSV  = "grades_all_runs.csv"

# ==== УТИЛИТЫ ====
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def now_ts() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def parse_patterns(patterns: str) -> List[str]:
    return [p.strip() for p in patterns.split(",") if p.strip()]

def discover_files(folder: Path, patterns: List[str]) -> List[Path]:
    files = []
    for pat in patterns:
        files.extend(folder.glob(pat))
    files = [f for f in files if f.is_file()]
    files.sort()
    return files

def sanitize_text(s: str, max_len: int = 100000) -> str:
    if s is None: return ""
    s = str(s)
    if len(s) > max_len:
        return s[:max_len] + f"\n...[output truncated at {max_len} chars]"
    return s

def to_iso(epoch: float) -> str:
    try:
        return datetime.fromtimestamp(float(epoch), tz=timezone.utc).isoformat()
    except Exception:
        return ""

def safe_slug(s: Optional[str]) -> str:
    s = (s or "").strip()
    s = re.sub(r"[^\w\-.]+", "_", s, flags=re.UNICODE)
    return s[:80] or "unknown"

def append_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]):
    ensure_dir(path.parent)
    file_exists = path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            w.writeheader()
        for r in rows:
            # гарантируем наличие всех полей
            w.writerow({k: r.get(k, "") for k in fieldnames})

# ==== ПАРСИНГ ПОСЛЕДНЕЙ СТРОКИ ====
def try_parse_json(s: str) -> Optional[Dict[str, Any]]:
    s = (s or "").strip()
    if not s: return None
    # Попробуем взять последний блок {...}
    if "{" in s and "}" in s:
        start = s.rfind("{"); end = s.rfind("}")
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(s[start:end+1])
            except Exception:
                pass
    try:
        return json.loads(s)
    except Exception:
        return None

def get_last_nonempty_line(text: str) -> str:
    for ln in reversed((text or "").splitlines()):
        if ln.strip(): return ln.strip()
    return ""

def parse_last_payload(last_text: str) -> Dict[str, Any]:
    """
    Пытаемся достать name/group/assignment/score из последней строки.
    Приоритет — JSON, иначе вернём то, что можем (name/score из K:V/двух строк).
    """
    payload = {"name": None, "group": None, "assignment": None, "score": None}
    js = try_parse_json(last_text)
    if isinstance(js, dict):
        # нормализуем ключи
        lower = {str(k).lower(): k for k in js.keys()}
        def pick(keys, cast=None):
            for kk in keys:
                if kk in lower:
                    v = js[lower[kk]]
                    if cast:
                        try: return cast(v)
                        except Exception: return None
                    return v
            return None
        payload["name"]       = pick(["name","имя","fio"], str)
        payload["group"]      = pick(["group","группа","grp"], str)
        payload["assignment"] = pick(["assignment","task","id","ид"], str)
        sc = pick(["score","баллы","points","балл"])
        if isinstance(sc, str):
            sc = sc.replace(",", ".")
            try: sc = float(sc)
            except Exception: sc = None
        if isinstance(sc, (int,float)):
            payload["score"] = float(sc)
        return payload

    # Fallback: K:V и/или две строки (как раньше)
    name_pat  = re.compile(r"(?im)^(?:name|имя|fio)\s*[:=]\s*(.+)$")
    group_pat = re.compile(r"(?im)^(?:group|группа|grp)\s*[:=]\s*(.+)$")
    asgn_pat  = re.compile(r"(?im)^(?:assignment|task|id|ид)\s*[:=]\s*(.+)$")
    score_pat = re.compile(r"(?im)^(?:score|баллы|points|балл)\s*[:=]\s*([0-9]+(?:[.,][0-9]+)?)$")

    m = name_pat.search(last_text);  payload["name"] = m.group(1).strip() if m else None
    m = group_pat.search(last_text); payload["group"] = m.group(1).strip() if m else None
    m = asgn_pat.search(last_text);  payload["assignment"]= m.group(1).strip() if m else None
    m = score_pat.search(last_text)
    if m:
        raw = m.group(1).replace(",", ".")
        try: payload["score"] = float(raw)
        except Exception: payload["score"] = None
        return payload

    # две строки: ... name \n score
    lines = [ln.strip() for ln in last_text.splitlines() if ln.strip()]
    if len(lines) >= 2:
        try:
            payload["score"] = float(lines[-1].replace(",", "."))
            payload["name"]  = lines[-2]
        except Exception:
            pass
    return payload

# ==== ВЫПОЛНЕНИЕ ФАЙЛОВ ====
NBCLIENT_AVAILABLE = True
try:
    import nbformat
    from nbclient import NotebookClient
    from nbclient.exceptions import CellExecutionError
except Exception:
    NBCLIENT_AVAILABLE = False

def run_python_script(file_path: Path, timeout: int) -> Dict[str, Any]:
    start = time.time()
    env = os.environ.copy()
    env.setdefault("CHECK_MODE","1")
    # пробрасываем mtime
    try: env["HARN_MTIME_EPOCH"] = str(int(file_path.stat().st_mtime))
    except Exception: pass
    try:
        proc = subprocess.run(
            [sys.executable, str(file_path)],
            cwd=str(file_path.parent),
            capture_output=True, text=True, timeout=timeout, env=env
        )
        elapsed = time.time() - start
        stdout = sanitize_text(proc.stdout)
        last_text = get_last_nonempty_line(stdout)
        payload = parse_last_payload(last_text) if last_text else {"name":None,"group":None,"assignment":None,"score":None}
        return {
            "status": "PASSED" if proc.returncode == 0 else "FAILED",
            "exit_code": proc.returncode,
            "stdout": stdout,
            "stderr": sanitize_text(proc.stderr),
            "duration_sec": round(elapsed,3),
            "last_output_text": last_text,
            "payload": payload,
        }
    except subprocess.TimeoutExpired as e:
        return {"status":"TIMEOUT","exit_code":None,"stdout":sanitize_text(e.stdout),
                "stderr":sanitize_text(e.stderr or f"Timeout after {timeout}"),
                "duration_sec":round(time.time()-start,3),
                "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}
    except Exception:
        return {"status":"ERROR","exit_code":None,"stdout":"","stderr":sanitize_text(traceback.format_exc()),
                "duration_sec":round(time.time()-start,3),
                "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}

def extract_last_output_from_nb(nb) -> str:
    last_code_idx = None
    for i in range(len(nb.cells)-1, -1, -1):
        if nb.cells[i].cell_type == "code":
            last_code_idx = i; break
    if last_code_idx is None: return ""
    cell = nb.cells[last_code_idx]
    outs = []
    for out in cell.get("outputs", []):
        ot = out.get("output_type")
        if ot == "stream":
            outs.append(out.get("text",""))
        elif ot in ("execute_result","display_data"):
            data = out.get("data", {})
            if "text/plain" in data:
                outs.append(str(data["text/plain"]))
            elif "application/json" in data:
                try: outs.append(json.dumps(data["application/json"], ensure_ascii=False))
                except Exception: outs.append(str(data["application/json"]))
        elif ot == "error":
            outs.append(f"[ERROR] {out.get('ename','Error')}: {out.get('evalue','')}")
    return "\n".join([t for t in outs if str(t).strip()])

def run_notebook(file_path: Path, timeout: int) -> Dict[str, Any]:
    start = time.time()
    if NBCLIENT_AVAILABLE:
        try:
            nb = nbformat.read(file_path, as_version=4)
            # пробрасываем mtime для ядра
            try: os.environ["HARN_MTIME_EPOCH"] = str(int(file_path.stat().st_mtime))
            except Exception: pass
            client = NotebookClient(
                nb, timeout=timeout, kernel_name='python3',
                resources={'metadata': {'path': str(file_path.parent)}}
            )
            client.execute()
            elapsed = time.time() - start
            # полный stdout-like
            outputs_text = []
            for cell in nb.cells:
                if cell.cell_type == "code":
                    for out in cell.get("outputs", []):
                        if out.get("output_type") == "stream":
                            outputs_text.append(out.get("text",""))
                        elif out.get("output_type") in ("execute_result","display_data"):
                            data = out.get("data",{})
                            if "text/plain" in data:
                                outputs_text.append(str(data["text/plain"]))
                            elif "application/json" in data:
                                try: outputs_text.append(json.dumps(data["application/json"], ensure_ascii=False))
                                except Exception: outputs_text.append(str(data["application/json"]))
            last_text = extract_last_output_from_nb(nb)
            payload = parse_last_payload(last_text) if last_text else {"name":None,"group":None,"assignment":None,"score":None}
            return {
                "status":"PASSED","exit_code":0,
                "stdout":sanitize_text("".join(outputs_text)),
                "stderr":"", "duration_sec":round(elapsed,3),
                "last_output_text":sanitize_text(last_text),
                "payload": payload,
            }
        except CellExecutionError as e:
            return {"status":"FAILED","exit_code":1,"stdout":"","stderr":sanitize_text(str(e)),
                    "duration_sec":round(time.time()-start,3),
                    "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}
        except Exception:
            return {"status":"ERROR","exit_code":None,"stdout":"","stderr":sanitize_text(traceback.format_exc()),
                    "duration_sec":round(time.time()-start,3),
                    "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}
    # Fallback: nbconvert
    try:
        try: os.environ["HARN_MTIME_EPOCH"] = str(int(file_path.stat().st_mtime))
        except Exception: pass
        tmp_out = file_path.parent / f"{file_path.stem}.executed.ipynb"
        proc = subprocess.run(
            ["jupyter","nbconvert","--to","notebook","--execute",str(file_path),"--output",str(tmp_out.name)],
            cwd=str(file_path.parent), capture_output=True, text=True, timeout=timeout
        )
        elapsed = time.time() - start
        status = "PASSED" if proc.returncode == 0 else "FAILED"
        last_text = ""
        if tmp_out.exists():
            try:
                nb2 = nbformat.read(tmp_out, as_version=4)
                last_text = extract_last_output_from_nb(nb2)
            except Exception: last_text = ""
        payload = parse_last_payload(last_text) if last_text else {"name":None,"group":None,"assignment":None,"score":None}
        return {"status":status,"exit_code":proc.returncode,"stdout":sanitize_text(proc.stdout),
                "stderr":sanitize_text(proc.stderr),"duration_sec":round(elapsed,3),
                "last_output_text":sanitize_text(last_text),"payload":payload}
    except subprocess.TimeoutExpired as e:
        return {"status":"TIMEOUT","exit_code":None,"stdout":sanitize_text(e.stdout),
                "stderr":sanitize_text(e.stderr or f"Timeout after {timeout}s"),
                "duration_sec":round(time.time()-start,3),
                "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}
    except Exception:
        return {"status":"ERROR","exit_code":None,"stdout":"","stderr":sanitize_text(traceback.format_exc()),
                "duration_sec":round(time.time()-start,3),
                "last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}

# ==== ОСНОВНОЙ РАННЕР ====
def run_checks(
    target_dir: Path,
    patterns: List[str],
    default_timeout: int,
    stop_on_fail: bool,
    reports_root: Path,
    save_logs: bool,
    verbosity: int = 1
) -> Dict[str, Any]:
    target_dir = target_dir.resolve()
    ensure_dir(target_dir)
    ensure_dir(reports_root)

    ts = now_ts()
    report_dir = reports_root / ts
    logs_dir = report_dir / "logs"
    subs_dir = reports_root / "submissions" / ts  # <-- сюда копируем работы
    ensure_dir(report_dir); ensure_dir(logs_dir); ensure_dir(subs_dir)

    files = discover_files(target_dir, patterns)
    if verbosity: print(f"[+] Found {len(files)} file(s) in {target_dir} by {patterns}")

    summary_rows = []
    grades_rows  = []
    all_summary_rows = []
    all_grades_rows  = []

    total_start = time.time()

    for idx, fpath in enumerate(files, start=1):
        rel_name = str(fpath.relative_to(target_dir))
        timeout = int(default_timeout)
        if verbosity: print(f"[{idx}/{len(files)}] Running: {rel_name} (timeout={timeout}s)")

        # --- запуск ---
        if fpath.suffix.lower() == ".py":
            result = run_python_script(fpath, timeout=timeout)
        elif fpath.suffix.lower() == ".ipynb":
            result = run_notebook(fpath, timeout=timeout)
        else:
            result = {"status":"SKIPPED","exit_code":None,"stdout":"","stderr":f"Unsupported extension: {fpath.suffix}",
                      "duration_sec":0.0,"last_output_text":"","payload":{"name":None,"group":None,"assignment":None,"score":None}}

        # --- копирование исходника в submissions/<ts>/...
        dest = subs_dir / rel_name
        ensure_dir(dest.parent)
        try:
            shutil.copy2(fpath, dest)
        except Exception as e:
            if verbosity: print(f"[WARN] Copy failed for {rel_name}: {e}")

        # --- логи (по месту запуска)
        if save_logs:
            base = logs_dir / (rel_name.replace(os.sep, "__"))
            with open(base.with_suffix(".stdout.txt"), "w", encoding="utf-8") as out_f:
                out_f.write(result.get("stdout","") or "")
            with open(base.with_suffix(".stderr.txt"), "w", encoding="utf-8") as err_f:
                err_f.write(result.get("stderr","") or "")
            with open(base.with_suffix(".last.txt"), "w", encoding="utf-8") as last_f:
                last_f.write(result.get("last_output_text","") or "")

        # --- mtime
        try:
            mtime_epoch = float(fpath.stat().st_mtime)
        except Exception:
            mtime_epoch = None
        mtime_iso = to_iso(mtime_epoch) if mtime_epoch else ""

        # --- строки отчётов
        status = result.get("status")
        row_sum = {
            "timestamp": ts,
            "file": rel_name,
            "status": status,
            "exit_code": result.get("exit_code"),
            "duration_sec": result.get("duration_sec"),
            "mtime_epoch": mtime_epoch or "",
            "mtime_iso": mtime_iso,
            "report_dir": str(report_dir),
        }
        summary_rows.append(row_sum)

        payload = result.get("payload") or {}
        row_grade = {
            "timestamp": ts,
            "file": rel_name,
            "name": payload.get("name"),
            "group": payload.get("group"),
            "assignment": payload.get("assignment"),
            "score": payload.get("score"),
            "status": status,
            "mtime_iso": mtime_iso,
            "report_dir": str(report_dir),
        }
        grades_rows.append(row_grade)

        # для агрегаторов (добавим те же поля)
        all_summary_rows.append(row_sum.copy())
        all_grades_rows.append(row_grade.copy())

        if stop_on_fail and status not in ("PASSED", "SKIPPED"):
            if verbosity: print("[!] Stopping on first failure as requested.")
            break

    total_elapsed = round(time.time() - total_start, 3)

    # --- per-run JSON/CSV ---
    with open(report_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump({"created_at": ts, "target_dir": str(target_dir),
                   "patterns": parse_patterns(GLOB_PATTERNS),
                   "total_elapsed_sec": total_elapsed, "items": summary_rows},
                  f, ensure_ascii=False, indent=2)
    with open(report_dir / "grades.json", "w", encoding="utf-8") as f:
        json.dump({"created_at": ts, "items": grades_rows}, f, ensure_ascii=False, indent=2)

    # CSV per-run
    sum_fields = ["timestamp","file","status","exit_code","duration_sec","mtime_epoch","mtime_iso","report_dir"]
    grd_fields = ["timestamp","file","name","group","assignment","score","status","mtime_iso","report_dir"]

    with open(report_dir / "summary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=sum_fields); w.writeheader()
        for r in summary_rows: w.writerow(r)

    with open(report_dir / "grades.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=grd_fields); w.writeheader()
        for r in grades_rows: w.writerow(r)

    # --- АГРЕГАТОРЫ В КОРНЕ REPORTS_ROOT: append ---
    append_csv(REPORTS_ROOT / ALL_SUMMARY_CSV, all_summary_rows, sum_fields)
    append_csv(REPORTS_ROOT / ALL_GRADES_CSV,  all_grades_rows,  grd_fields)

    if VERBOSITY:
        print(f"\n[✓] Done in {total_elapsed}s. Report: {report_dir}")
        print(f"    Copied submissions to: {subs_dir}")
        print(f"    Updated aggregators: {REPORTS_ROOT/ALL_SUMMARY_CSV}, {REPORTS_ROOT/ALL_GRADES_CSV}")

    return {
        "report_dir": str(report_dir),
        "summary_rows": summary_rows,
        "grades_rows": grades_rows
    }

# ==== UI (если ipywidgets установлен) ====
WIDGETS_AVAILABLE = False
#try:
#    import ipywidgets as widgets
#    from IPython.display import display
#except Exception:
#    WIDGETS_AVAILABLE = False
#    def display(x): print(x)

if WIDGETS_AVAILABLE:
    folder_input   = widgets.Text(value=str(TARGET_DIR), description='Folder:', layout=widgets.Layout(width='60%'))
    patterns_input = widgets.Text(value=str(GLOB_PATTERNS), description='Patterns:')
    timeout_input  = widgets.IntText(value=int(DEFAULT_TIMEOUT), description='Timeout:')
    stopfail_input = widgets.Checkbox(value=bool(STOP_ON_FAIL), description='Stop on fail')
    savelogs_input = widgets.Checkbox(value=bool(SAVE_LOGS), description='Save logs')
    verbosity_input= widgets.IntSlider(value=int(VERBOSITY), min=0, max=2, step=1, description='Verbosity')

    run_btn = widgets.Button(description='Run checks', button_style='primary')
    out = widgets.Output()

    def on_run_clicked(_):
        out.clear_output()
        with out:
            td = Path(folder_input.value)
            pats = [p.strip() for p in patterns_input.value.split(',') if p.strip()]
            res = run_checks(td, pats, int(timeout_input.value), bool(stopfail_input.value),
                             REPORTS_ROOT, bool(savelogs_input.value), int(verbosity_input.value))
            print("\nReport:", res.get("report_dir"))

    run_btn.on_click(on_run_clicked)
    ui = widgets.VBox([
        widgets.HBox([folder_input]),
        widgets.HBox([patterns_input]),
        widgets.HBox([timeout_input, stopfail_input, savelogs_input, verbosity_input]),
        run_btn,
        out
    ])
    display(ui)
else:
    print("[INFO] ipywidgets not available. Use the call below to run without UI.")

# ▶️ Запуск без UI (раскомментируй при желании)
run_checks(TARGET_DIR, parse_patterns(GLOB_PATTERNS), DEFAULT_TIMEOUT, STOP_ON_FAIL, REPORTS_ROOT, SAVE_LOGS, VERBOSITY)


[INFO] ipywidgets not available. Use the call below to run without UI.
[+] Found 18 file(s) in /mnt/d/projects/home-work-checker/data by ['*.py', '*.ipynb']
[1/18] Running: Evans_Owamoyo_Task0_Numpy.ipynb (timeout=300s)
[2/18] Running: Task0_Numpy_Asadullin.ipynb (timeout=300s)
[3/18] Running: Task0_Numpy_Bedengov_11-451.ipynb (timeout=300s)
[4/18] Running: Task0_Numpy_Boraik.ipynb (timeout=300s)
[5/18] Running: Task0_Numpy_Budrevich.ipynb (timeout=300s)
[6/18] Running: Task0_Numpy_Gareev_KI.ipynb (timeout=300s)
[7/18] Running: Task0_Numpy_Gubaydullin.ipynb (timeout=300s)
[8/18] Running: Task0_Numpy_Iliasova_11_451.ipynb (timeout=300s)
[9/18] Running: Task0_Numpy_Islamov_11_451_.ipynb (timeout=300s)
[10/18] Running: Task0_Numpy_Kabyshev.ipynb (timeout=300s)
[11/18] Running: Task0_Numpy_Korobov.ipynb (timeout=300s)
[12/18] Running: Task0_Numpy_Muradymov.ipynb (timeout=300s)
[13/18] Running: Task0_Numpy_Nemanov.ipynb (timeout=300s)
[14/18] Running: Task0_Numpy_Sadriev.ipynb (timeout=300s

{'report_dir': 'reports/20250916_170455',
 'summary_rows': [{'timestamp': '20250916_170455',
   'file': 'Evans_Owamoyo_Task0_Numpy.ipynb',
   'status': 'PASSED',
   'exit_code': 0,
   'duration_sec': 1.411,
   'mtime_epoch': 1758031382.4149687,
   'mtime_iso': '2025-09-16T14:03:02.414969+00:00',
   'report_dir': 'reports/20250916_170455'},
  {'timestamp': '20250916_170455',
   'file': 'Task0_Numpy_Asadullin.ipynb',
   'status': 'PASSED',
   'exit_code': 0,
   'duration_sec': 1.107,
   'mtime_epoch': 1758031388.6156108,
   'mtime_iso': '2025-09-16T14:03:08.615611+00:00',
   'report_dir': 'reports/20250916_170455'},
  {'timestamp': '20250916_170455',
   'file': 'Task0_Numpy_Bedengov_11-451.ipynb',
   'status': 'PASSED',
   'exit_code': 0,
   'duration_sec': 0.864,
   'mtime_epoch': 1758031397.9466221,
   'mtime_iso': '2025-09-16T14:03:17.946622+00:00',
   'report_dir': 'reports/20250916_170455'},
  {'timestamp': '20250916_170455',
   'file': 'Task0_Numpy_Boraik.ipynb',
   'status': 'PASS